# LLM Steel Defect Classification — Claude + Groq Ensemble

This notebook keeps the same output style as your previous notebook: progress logs, saved JSON, detection visualizations, area-wise charts, accuracy summary chart, confusion matrix, misclassified samples, and single-image demo.

It is **LLM-based only**: Claude + Groq are used for classification, with handcrafted image features, rule-prior evidence, weighted voting, caching, and XML/LLM/contour area detection.


In [1]:

# ================================================================
# CELL 1 — SETUP
# ================================================================
import os, cv2, json, math, time, warnings, hashlib, re
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from collections import Counter, defaultdict
from dotenv import load_dotenv
warnings.filterwarnings("ignore")

try:
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
    SKIMAGE_OK = True
except Exception:
    SKIMAGE_OK = False
    print("skimage not found — GLCM/LBP features disabled")

try:
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False
    print("sklearn not found — confusion matrix/report disabled")

try:
    from groq import Groq
except Exception:
    Groq = None
    print("groq package missing. Install: pip install groq")

try:
    import anthropic
except Exception:
    anthropic = None
    print("anthropic package missing. Install: pip install anthropic")

load_dotenv()

groq_client = None
claude_client = None

groq_key = os.getenv("GROQ_API_KEY")
claude_key = os.getenv("ANTHROPIC_API_KEY")

print("="*74)
print("  LLM STEEL DEFECT CLASSIFICATION — CLAUDE + GROQ ENSEMBLE")
print("="*74)

if Groq and groq_key:
    groq_client = Groq(api_key=groq_key)
    print(f"Groq   READY  ({groq_key[:10]}...)")
else:
    print("Groq   NOT READY — set GROQ_API_KEY in .env")

if anthropic and claude_key:
    claude_client = anthropic.Anthropic(api_key=claude_key)
    print(f"Claude READY  ({claude_key[:10]}...)")
else:
    print("Claude NOT READY — set ANTHROPIC_API_KEY in .env")

print("="*74)

if not groq_client and not claude_client:
    raise RuntimeError("At least one key required: GROQ_API_KEY or ANTHROPIC_API_KEY")


  LLM STEEL DEFECT CLASSIFICATION — CLAUDE + GROQ ENSEMBLE
Groq   READY  (gsk_YUlv7A...)
Claude READY  (sk-ant-api...)


In [2]:

# ================================================================
# CELL 2 — CONFIGURATION
# ================================================================
CLASS_NAMES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

BASE_PATH       = r"C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\NEU-DET"
IMAGES_DIR      = os.path.join(BASE_PATH, "IMAGES")
ANNOTATIONS_DIR = os.path.join(BASE_PATH, "ANNOTATIONS")
OUTPUT_DIR      = os.path.join(BASE_PATH, "OUTPUT_LLM_CLAUDE_GROQ")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_JSON    = os.path.join(OUTPUT_DIR, "llm_claude_groq_results_all.json")
CACHE_JSON      = os.path.join(OUTPUT_DIR, "llm_vote_cache.json")
ERROR_JSON      = os.path.join(OUTPUT_DIR, "llm_errors.json")

IMG_SIZE        = 200
MODEL_NAME      = "LLM_Claude_Groq_Steel_Ensemble"
USE_LLM_AREA    = True
SAVE_EVERY      = 25
PRINT_EVERY     = 50

# FAST      = fewer calls, lower cost
# BALANCED  = recommended
# MAX       = strongest LLM ensemble but expensive and slow
RUN_MODE = "BALANCED"

# To test first, set LIMIT_IMAGES = 60. For full dataset, set None.
LIMIT_IMAGES = None

DEFECT_COLORS = {
    "crazing":        "#FF6B6B",
    "inclusion":      "#4ECDC4",
    "patches":        "#45B7D1",
    "pitted_surface": "#96CEB4",
    "rolled-in_scale":"#FFEAA7",
    "scratches":      "#DDA0DD"
}
SOURCE_COLORS = {
    "xml_annotation":      "#00FF88",
    "llm_groq":            "#4FC3F7",
    "llm_claude":          "#FF8A65",
    "contour_detection":   "#FFA726",
    "full_image_fallback": "#EF5350"
}

print(f"BASE_PATH   : {BASE_PATH}")
print(f"IMAGES_DIR  : {IMAGES_DIR}")
print(f"ANNOTATIONS : {ANNOTATIONS_DIR}")
print(f"OUTPUT      : {OUTPUT_DIR}")
print(f"RUN_MODE    : {RUN_MODE}")


BASE_PATH   : C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\NEU-DET
IMAGES_DIR  : C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\NEU-DET\IMAGES
ANNOTATIONS : C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\NEU-DET\ANNOTATIONS
OUTPUT      : C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\NEU-DET\OUTPUT_LLM_CLAUDE_GROQ
RUN_MODE    : BALANCED


In [3]:

# ================================================================
# CELL 3 — RICH IMAGE FEATURE EXTRACTION
# ================================================================
def safe_round(x, n=3):
    try:
        if x is None or np.isnan(x) or np.isinf(x):
            return 0
        return round(float(x), n)
    except Exception:
        return 0

def compute_glcm(gray_u8):
    if not SKIMAGE_OK:
        return {}
    try:
        g = (gray_u8 // 32).astype(np.uint8)
        glcm = graycomatrix(g, [1, 2], [0, np.pi/4, np.pi/2, 3*np.pi/4], levels=8, symmetric=True, normed=True)
        return {
            "glcm_contrast":    safe_round(np.mean(graycoprops(glcm, "contrast")), 3),
            "glcm_homogeneity": safe_round(np.mean(graycoprops(glcm, "homogeneity")), 3),
            "glcm_energy":      safe_round(np.mean(graycoprops(glcm, "energy")), 3),
            "glcm_correlation": safe_round(np.mean(graycoprops(glcm, "correlation")), 3),
        }
    except Exception:
        return {}

def compute_lbp(gray_u8):
    if not SKIMAGE_OK:
        return {}
    try:
        lbp = local_binary_pattern(gray_u8, P=16, R=2, method="uniform")
        hist, _ = np.histogram(lbp.ravel(), bins=20, range=(0, 20), density=True)
        return {
            "lbp_uniformity": safe_round(np.max(hist), 4),
            "lbp_entropy": safe_round(-np.sum(hist * np.log2(hist + 1e-9)), 3)
        }
    except Exception:
        return {}

def contour_region_features(gray_u8):
    H, W = gray_u8.shape
    blur = cv2.GaussianBlur(gray_u8, (5,5), 0)
    th1 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY_INV, 31, 4)
    _, th2 = cv2.threshold(blur, int(np.percentile(blur, 18)), 255, cv2.THRESH_BINARY_INV)
    mask = cv2.bitwise_or(th1, th2)
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    areas, aspects, circs, boxes = [], [], [], []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 8:
            continue
        x,y,w,h = cv2.boundingRect(cnt)
        if w <= 1 or h <= 1:
            continue
        peri = cv2.arcLength(cnt, True)
        circ = 4*np.pi*area/(peri*peri + 1e-6)
        asp = max(w/h, h/w)
        areas.append(area)
        aspects.append(asp)
        circs.append(circ)
        boxes.append((x,y,w,h,area,asp,circ))

    total_area = H*W
    return {
        "num_regions": len(areas),
        "avg_area": safe_round(np.mean(areas) if areas else 0, 2),
        "max_area": safe_round(np.max(areas) if areas else 0, 2),
        "total_defect_pct": safe_round(sum(areas)/total_area*100 if areas else 0, 2),
        "avg_aspect_ratio": safe_round(np.mean(aspects) if aspects else 1, 3),
        "max_aspect_ratio": safe_round(np.max(aspects) if aspects else 1, 3),
        "avg_circularity": safe_round(np.mean(circs) if circs else 0, 3),
        "min_circularity": safe_round(np.min(circs) if circs else 0, 3),
        "regions_raw": boxes[:20]
    }

def extract_rich_features(img_path, img_size=200):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None, None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (img_size, img_size))
    gray_u8 = cv2.cvtColor(img_res, cv2.COLOR_RGB2GRAY)
    gray = gray_u8.astype(np.float32)
    H, W = gray.shape

    f = {}
    f["mean_intensity"] = safe_round(np.mean(gray), 2)
    f["std_intensity"]  = safe_round(np.std(gray), 2)
    f["contrast"]       = safe_round(np.std(gray)/(np.mean(gray)+1e-5), 3)
    f["dark_pixel_pct"] = safe_round(np.mean(gray < np.percentile(gray, 20))*100, 2)
    f["bright_pixel_pct"] = safe_round(np.mean(gray > np.percentile(gray, 80))*100, 2)

    hist = np.histogram(gray_u8, bins=256, range=(0,256))[0].astype(float)
    p = hist / (hist.sum() + 1e-9)
    p2 = p[p > 0]
    f["entropy"] = safe_round(-np.sum(p2*np.log2(p2)), 3)

    edges = cv2.Canny(gray_u8, 40, 140)
    gx = cv2.Sobel(gray_u8, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_u8, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    ang = np.arctan2(np.abs(gy), np.abs(gx) + 1e-6)

    f["edge_density"] = safe_round(np.mean(edges > 0)*100, 2)
    f["sobel_magnitude"] = safe_round(np.mean(mag), 3)
    f["horizontal_edge_ratio"] = safe_round(np.mean(ang < np.pi/6), 3)
    f["vertical_edge_ratio"] = safe_round(np.mean(ang > np.pi/3), 3)
    f["diagonal_edge_ratio"] = safe_round(np.mean((ang >= np.pi/6) & (ang <= np.pi/3)), 3)

    # Line detection evidence
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=25, minLineLength=25, maxLineGap=8)
    line_lengths, line_angles = [], []
    if lines is not None:
        for l in lines[:80]:
            x1,y1,x2,y2 = l[0]
            length = np.hypot(x2-x1, y2-y1)
            angle = abs(np.degrees(np.arctan2(y2-y1, x2-x1)))
            if angle > 90: angle = 180 - angle
            line_lengths.append(length)
            line_angles.append(angle)
    f["hough_line_count"] = len(line_lengths)
    f["avg_line_length"] = safe_round(np.mean(line_lengths) if line_lengths else 0, 2)
    f["max_line_length"] = safe_round(np.max(line_lengths) if line_lengths else 0, 2)
    f["line_horizontal_pct"] = safe_round(np.mean(np.array(line_angles) < 20)*100 if line_angles else 0, 2)

    # Quadrant evidence
    q_std, q_edge = {}, {}
    for name, sl in {
        "top_left": (slice(0,H//2), slice(0,W//2)),
        "top_right": (slice(0,H//2), slice(W//2,W)),
        "bottom_left": (slice(H//2,H), slice(0,W//2)),
        "bottom_right": (slice(H//2,H), slice(W//2,W)),
    }.items():
        q = gray[sl]
        e = edges[sl]
        q_std[name] = safe_round(np.std(q), 2)
        q_edge[name] = safe_round(np.mean(e > 0)*100, 2)
    f["quadrant_std"] = q_std
    f["quadrant_edge_density"] = q_edge

    f.update(compute_glcm(gray_u8))
    f.update(compute_lbp(gray_u8))
    f.update(contour_region_features(gray_u8))

    return f, img_res

print("Cell 3 OK — rich feature extraction defined")


Cell 3 OK — rich feature extraction defined


In [4]:

# ================================================================
# CELL 4 — RULE PRIOR + STRICT LLM PROMPTS
# ================================================================
def rule_prior_scores(f):
    s = {c: 0.0 for c in CLASS_NAMES}
    n = f.get("num_regions",0)
    max_asp = f.get("max_aspect_ratio",1)
    avg_asp = f.get("avg_aspect_ratio",1)
    min_circ = f.get("min_circularity",0)
    avg_circ = f.get("avg_circularity",0)
    max_area = f.get("max_area",0)
    avg_area = f.get("avg_area",0)
    coverage = f.get("total_defect_pct",0)
    entropy = f.get("entropy",0)
    contrast = f.get("contrast",0)
    dark = f.get("dark_pixel_pct",0)
    edge = f.get("edge_density",0)
    h_ratio = f.get("horizontal_edge_ratio",0)
    line_count = f.get("hough_line_count",0)
    max_line = f.get("max_line_length",0)
    line_h = f.get("line_horizontal_pct",0)

    # scratches: thin elongated line evidence
    s["scratches"] += 3.0 if max_asp >= 5.5 else 0
    s["scratches"] += 1.5 if min_circ < 0.18 else 0
    s["scratches"] += 1.3 if max_line > 55 else 0
    s["scratches"] += 0.8 if line_count >= 3 else 0

    # rolled-in_scale: elongated horizontal scale-like defects
    s["rolled-in_scale"] += 2.4 if avg_asp >= 2.4 else 0
    s["rolled-in_scale"] += 1.6 if line_h >= 45 or h_ratio >= 0.48 else 0
    s["rolled-in_scale"] += 1.0 if coverage >= 6 and coverage <= 25 else 0

    # patches: broad irregular areas
    s["patches"] += 3.0 if max_area >= 1800 else 0
    s["patches"] += 2.0 if coverage >= 18 else 0
    s["patches"] += 1.2 if n <= 5 and max_area >= 1000 else 0

    # pitted: many small circular regions
    s["pitted_surface"] += 2.5 if n >= 12 else 0
    s["pitted_surface"] += 2.0 if avg_area < 130 and avg_circ > 0.50 else 0
    s["pitted_surface"] += 0.8 if edge >= 8 else 0

    # crazing: web-like high entropy cracks
    s["crazing"] += 2.0 if entropy >= 6.2 else 0
    s["crazing"] += 1.5 if 5 <= n <= 25 else 0
    s["crazing"] += 1.0 if edge >= 10 and contrast >= 0.20 else 0

    # inclusion: dark embedded particles, medium/small isolated areas
    s["inclusion"] += 1.8 if dark >= 18 else 0
    s["inclusion"] += 1.4 if 1 <= n <= 10 and max_area < 1800 else 0
    s["inclusion"] += 1.0 if avg_circ >= 0.35 and avg_asp < 3.0 else 0

    total = sum(max(v,0) for v in s.values()) + 1e-9
    return {k: round(v/total, 4) for k,v in s.items()}

def compact_features_text(f):
    prior = rule_prior_scores(f)
    top_prior = sorted(prior.items(), key=lambda x: x[1], reverse=True)
    return f"""
FEATURES:
regions_count={f.get('num_regions',0)}
avg_area={f.get('avg_area',0)} max_area={f.get('max_area',0)} total_defect_pct={f.get('total_defect_pct',0)}
avg_aspect_ratio={f.get('avg_aspect_ratio',1)} max_aspect_ratio={f.get('max_aspect_ratio',1)}
avg_circularity={f.get('avg_circularity',0)} min_circularity={f.get('min_circularity',0)}
entropy={f.get('entropy',0)} contrast={f.get('contrast',0)} edge_density={f.get('edge_density',0)}
horizontal_edge_ratio={f.get('horizontal_edge_ratio',0)} vertical_edge_ratio={f.get('vertical_edge_ratio',0)}
hough_line_count={f.get('hough_line_count',0)} max_line_length={f.get('max_line_length',0)} line_horizontal_pct={f.get('line_horizontal_pct',0)}
dark_pixel_pct={f.get('dark_pixel_pct',0)} bright_pixel_pct={f.get('bright_pixel_pct',0)}
glcm_contrast={f.get('glcm_contrast',0)} glcm_homogeneity={f.get('glcm_homogeneity',0)} lbp_entropy={f.get('lbp_entropy',0)}
rule_prior={prior}
top_rule_prior={top_prior[:3]}
"""

def build_classify_prompt_v1(f):
    return f"""You are classifying steel surface defects from image-derived visual features.

Allowed labels only:
crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches

Decision hints:
- scratches: long, thin, very elongated defect line; high max aspect ratio.
- rolled-in_scale: elongated scale-like defect, often horizontal, strip-like.
- patches: broad irregular damaged area; high max area and coverage.
- pitted_surface: many small round pits; many regions, small area, circular.
- crazing: web-like crack network; high entropy and dense fine edges.
- inclusion: embedded dark particles/foreign material; isolated dark areas.

{compact_features_text(f)}

Return ONLY JSON:
{{"label":"one_allowed_label","confidence":0.0,"reason":"short evidence"}}"""

def build_classify_prompt_v2(f):
    return f"""Classify the defect. Use evidence, but output only strict JSON.

Important tie-breakers:
1. If line is extremely elongated, choose scratches.
2. If elongated and horizontal/scale-like, choose rolled-in_scale.
3. If one/few large regions dominate, choose patches.
4. If many tiny circular pits, choose pitted_surface.
5. If fine crack-web texture, choose crazing.
6. Otherwise choose inclusion.

{compact_features_text(f)}

JSON only:
{{"label":"crazing|inclusion|patches|pitted_surface|rolled-in_scale|scratches","confidence":0.0,"reason":"short"}}"""

def build_classify_prompt_v3(f):
    return f"""Steel defect classification using numerical descriptors.

Classes:
crazing = crack network
inclusion = dark embedded particles
patches = large irregular region
pitted_surface = many round pits
rolled-in_scale = rolled/strip/scale defect
scratches = thin scratch line

{compact_features_text(f)}

Give final answer in JSON only:
{{"label":"...", "confidence":0.0, "reason":"..."}}"""

def build_classify_prompt_v4(f):
    prior = rule_prior_scores(f)
    return f"""You are an expert industrial inspection assistant.
The rule-based prior is not final; use it only as evidence.

Rule prior probabilities:
{json.dumps(prior)}

Visual features:
{compact_features_text(f)}

Select exactly one label from:
{CLASS_NAMES}

Return JSON only:
{{"label":"...", "confidence":0.0, "reason":"..."}}"""

print("Cell 4 OK — rule prior and prompts defined")


Cell 4 OK — rule prior and prompts defined


In [5]:

# ================================================================
# CELL 5 — CLAUDE + GROQ WEIGHTED ENSEMBLE WITH CACHE
# ================================================================
# Current-style model IDs. If one model is unavailable in your account, code skips it safely.
CLAUDE_MODELS_BY_MODE = {
    "FAST": [
        ("claude-haiku-4-5", 0.88),
        ("claude-sonnet-4-6", 0.96),
    ],
    "BALANCED": [
        ("claude-sonnet-4-6", 1.00),
        ("claude-haiku-4-5", 0.88),
        ("claude-opus-4-7", 1.05),
    ],
    "MAX": [
        ("claude-opus-4-7", 1.08),
        ("claude-sonnet-4-6", 1.00),
        ("claude-haiku-4-5", 0.88),
    ]
}

GROQ_MODELS_BY_MODE = {
    "FAST": [
        ("llama-3.1-8b-instant", 0.78),
        ("llama-3.3-70b-versatile", 0.95),
    ],
    "BALANCED": [
        ("llama-3.3-70b-versatile", 0.98),
        ("openai/gpt-oss-120b", 0.94),
        ("qwen/qwen3-32b", 0.86),
    ],
    "MAX": [
        ("llama-3.3-70b-versatile", 0.98),
        ("openai/gpt-oss-120b", 0.94),
        ("qwen/qwen3-32b", 0.86),
        ("meta-llama/llama-4-scout-17b-16e-instruct", 0.90),
        ("llama-3.1-8b-instant", 0.78),
    ]
}

PROMPT_BUILDERS = [build_classify_prompt_v1, build_classify_prompt_v2, build_classify_prompt_v3, build_classify_prompt_v4]

def load_cache():
    if os.path.exists(CACHE_JSON):
        try:
            with open(CACHE_JSON, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(cache):
    with open(CACHE_JSON, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2)

LLM_CACHE = load_cache()
print(f"Cache loaded: {len(LLM_CACHE)} items")

def _feature_hash(f):
    slim = {k:v for k,v in f.items() if k != "regions_raw"}
    s = json.dumps(slim, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _parse_llm_json(text):
    if not text:
        return None, 0.0, ""
    raw = str(text).strip()
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            label = str(obj.get("label","")).strip().lower().replace("-", "_")
            conf = float(obj.get("confidence", 0.70))
            reason = str(obj.get("reason",""))[:160]
            for cls in CLASS_NAMES:
                if label == cls.replace("-", "_") or label == cls:
                    return cls, max(0.0, min(conf, 1.0)), reason
        except Exception:
            pass
    t = raw.lower().replace("-", "_")
    for cls in CLASS_NAMES:
        if cls.replace("-", "_") in t:
            return cls, 0.65, raw[:160]
    return None, 0.0, raw[:160]

def _call_claude(prompt, model, max_tokens=120, retries=2):
    if claude_client is None:
        return None
    for attempt in range(retries):
        try:
            resp = claude_client.messages.create(
                model=model,
                max_tokens=max_tokens,
                temperature=0,
                messages=[{"role":"user","content":prompt}]
            )
            return resp.content[0].text
        except Exception as e:
            err = str(e).lower()
            if any(x in err for x in ["rate", "429", "overload", "timeout"]):
                time.sleep(0.8 * (2 ** attempt))
                continue
            return None
    return None

def _call_groq(prompt, model, max_tokens=120, retries=2):
    if groq_client is None:
        return None
    for attempt in range(retries):
        try:
            resp = groq_client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}],
                temperature=0,
                max_tokens=max_tokens,
                response_format={"type":"json_object"}
            )
            return resp.choices[0].message.content
        except Exception as e:
            err = str(e).lower()
            if "response_format" in err or "json" in err:
                try:
                    resp = groq_client.chat.completions.create(
                        model=model,
                        messages=[{"role":"user","content":prompt}],
                        temperature=0,
                        max_tokens=max_tokens
                    )
                    return resp.choices[0].message.content
                except Exception:
                    return None
            if any(x in err for x in ["rate", "429", "timeout"]):
                time.sleep(0.8 * (2 ** attempt))
                continue
            return None
    return None

def classify_defect_ensemble(f, image_key=None):
    cache_key = image_key or _feature_hash(f)
    if cache_key in LLM_CACHE:
        cached = LLM_CACHE[cache_key]
        return cached["pred_label"], cached["confidence"], cached["method"], cached.get("votes", [])

    prompts = [pb(f) for pb in PROMPT_BUILDERS]
    votes = []
    weighted = defaultdict(float)

    # Add rule prior as a weak but useful voter
    prior = rule_prior_scores(f)
    for cls, p in prior.items():
        weighted[cls] += p * 0.70
    votes.append({"provider":"rule_prior","model":"rule_prior","label":max(prior,key=prior.get),
                  "confidence":prior[max(prior,key=prior.get)],"weight":0.70,"reason":"handcrafted feature prior"})

    claude_models = CLAUDE_MODELS_BY_MODE.get(RUN_MODE, CLAUDE_MODELS_BY_MODE["BALANCED"])
    groq_models   = GROQ_MODELS_BY_MODE.get(RUN_MODE, GROQ_MODELS_BY_MODE["BALANCED"])

    for model, mw in claude_models:
        for pi, prompt in enumerate(prompts):
            txt = _call_claude(prompt, model)
            label, conf, reason = _parse_llm_json(txt)
            if label:
                w = mw * max(conf, 0.50)
                weighted[label] += w
                votes.append({"provider":"claude","model":model,"prompt":pi+1,"label":label,
                              "confidence":round(conf,3),"weight":round(w,3),"reason":reason})

    for model, mw in groq_models:
        for pi, prompt in enumerate(prompts):
            txt = _call_groq(prompt, model)
            label, conf, reason = _parse_llm_json(txt)
            if label:
                w = mw * max(conf, 0.50)
                weighted[label] += w
                votes.append({"provider":"groq","model":model,"prompt":pi+1,"label":label,
                              "confidence":round(conf,3),"weight":round(w,3),"reason":reason})

    if not weighted:
        best = max(prior, key=prior.get)
        conf = 0.50 + 0.25 * prior[best]
        method = "RulePriorFallback"
        votes = [{"provider":"fallback","model":"rule_prior","label":best,"confidence":conf,"weight":1.0,"reason":"No LLM vote"}]
    else:
        best = max(weighted, key=weighted.get)
        total_w = sum(weighted.values()) + 1e-9
        agree = weighted[best] / total_w
        llm_vote_count = sum(1 for v in votes if v.get("label") == best and v.get("provider") in ["claude","groq"])
        provider_set = {v.get("provider") for v in votes if v.get("label") == best}
        both_bonus = 0.05 if ("claude" in provider_set and "groq" in provider_set) else 0
        conf = min(0.99, 0.58 + 0.34*agree + both_bonus + min(0.04, llm_vote_count*0.004))
        method = f"Claude+GroqWeighted({llm_vote_count}/{max(len(votes)-1,1)}LLM,best_w={weighted[best]:.2f})"

    record = {
        "pred_label": best,
        "confidence": round(float(conf), 4),
        "method": method,
        "weighted_scores": {k: round(v,4) for k,v in weighted.items()},
        "votes": votes
    }
    LLM_CACHE[cache_key] = record
    return best, float(conf), method, votes

print("Cell 5 OK — Claude + Groq weighted ensemble defined")


Cache loaded: 0 items
Cell 5 OK — Claude + Groq weighted ensemble defined


In [6]:

# ================================================================
# CELL 6 — AREA DETECTION: XML → CLAUDE/GROQ → CONTOUR
# ================================================================
def load_xml_annotations(img_filename, target_size=(200,200)):
    base = os.path.splitext(img_filename)[0]
    xml = os.path.join(ANNOTATIONS_DIR, base + ".xml")
    if not os.path.exists(xml):
        return None
    try:
        root = ET.parse(xml).getroot()
        sz = root.find("size")
        orig_w = int(sz.find("width").text)
        orig_h = int(sz.find("height").text)
        sx, sy = target_size[1]/orig_w, target_size[0]/orig_h
        regions = []
        for obj in root.findall("object"):
            bb = obj.find("bndbox")
            xmin = max(0, int(float(bb.find("xmin").text)*sx))
            ymin = max(0, int(float(bb.find("ymin").text)*sy))
            xmax = min(target_size[1], int(float(bb.find("xmax").text)*sx))
            ymax = min(target_size[0], int(float(bb.find("ymax").text)*sy))
            w, h = xmax-xmin, ymax-ymin
            label = obj.find("name").text if obj.find("name") is not None else "defect"
            if w > 2 and h > 2:
                regions.append({"bbox":[xmin,ymin,w,h], "label":label, "area":w*h, "source":"xml_annotation"})
        return regions or None
    except Exception:
        return None

def _build_area_prompt(f, pred_class, img_size):
    return f"""Find approximate defect bounding boxes using only numerical features.

Predicted defect class: {pred_class}
Image size: {img_size}x{img_size}

Use likely class shape:
- scratches: thin long line boxes
- rolled-in_scale: horizontal elongated boxes
- patches: large irregular broad boxes
- pitted_surface: multiple small pit boxes
- crazing: crack/network boxes, may be scattered
- inclusion: small/medium dark particle boxes

Features:
{compact_features_text(f)}

Return JSON only:
{{"regions":[{{"bbox":[x,y,w,h],"label":"{pred_class}","confidence":0.0}}]}}
Coordinates must be inside image. Return 1 to 5 best regions only.
"""

def _parse_bbox_json(text, img_size, pred_class):
    if not text:
        return None
    try:
        m = re.search(r"\{.*\}", str(text), flags=re.S)
        obj = json.loads(m.group(0) if m else text)
        regs = obj.get("regions", [])
        out = []
        for r in regs[:5]:
            bb = r.get("bbox", [])
            if len(bb) != 4:
                continue
            x,y,w,h = [int(float(v)) for v in bb]
            x = max(0, min(x, img_size-1))
            y = max(0, min(y, img_size-1))
            w = max(3, min(w, img_size-x))
            h = max(3, min(h, img_size-y))
            out.append({"bbox":[x,y,w,h], "label":pred_class, "area":w*h,
                        "confidence":float(r.get("confidence",0.70)), "source":"llm_bbox"})
        return out or None
    except Exception:
        return None

def contour_regions(img_res, pred_class, img_size=200):
    gray = cv2.cvtColor(img_res, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    if pred_class in ["scratches", "rolled-in_scale", "crazing"]:
        edges = cv2.Canny(blur, 35, 120)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,2))
        mask = cv2.dilate(edges, kernel, iterations=1)
    else:
        _, mask = cv2.threshold(blur, int(np.percentile(blur, 20)), 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    regs = []
    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        area = w*h
        if area < 25:
            continue
        asp = max(w/h, h/w)
        if pred_class == "scratches" and asp < 3:
            continue
        if pred_class == "pitted_surface" and area > 900:
            continue
        regs.append({"bbox":[x,y,w,h], "label":pred_class, "area":area, "source":"contour_detection"})

    regs = sorted(regs, key=lambda r: r["area"], reverse=True)[:8]
    if not regs:
        regs = [{"bbox":[10,10,img_size-20,img_size-20], "label":pred_class,
                 "area":(img_size-20)*(img_size-20), "source":"full_image_fallback"}]
    return regs

def detect_areas(f, img_res, pred_class, img_filename, img_size=200, use_llm=True):
    xml_regions = load_xml_annotations(img_filename, (img_size,img_size))
    if xml_regions:
        return xml_regions, "xml_annotation"

    if use_llm:
        prompt = _build_area_prompt(f, pred_class, img_size)
        if groq_client:
            txt = _call_groq(prompt, GROQ_MODELS_BY_MODE.get(RUN_MODE, GROQ_MODELS_BY_MODE["BALANCED"])[0][0], max_tokens=300)
            regs = _parse_bbox_json(txt, img_size, pred_class)
            if regs:
                for r in regs: r["source"] = "llm_groq"
                return regs, "llm_groq"

        if claude_client:
            txt = _call_claude(prompt, CLAUDE_MODELS_BY_MODE.get(RUN_MODE, CLAUDE_MODELS_BY_MODE["BALANCED"])[0][0], max_tokens=300)
            regs = _parse_bbox_json(txt, img_size, pred_class)
            if regs:
                for r in regs: r["source"] = "llm_claude"
                return regs, "llm_claude"

    return contour_regions(img_res, pred_class, img_size), "contour_detection"

print("Cell 6 OK — area detection defined")


Cell 6 OK — area detection defined


In [7]:

# ================================================================
# CELL 7 — SINGLE IMAGE PIPELINE
# ================================================================
def process_image(img_path, img_filename, true_label=None, use_llm_area=True, img_size=200):
    f, img_res = extract_rich_features(img_path, img_size)
    if f is None:
        return {"error": f"load failed: {img_path}"}

    image_key = f"{true_label or ''}/{img_filename}/{_feature_hash(f)}"
    pred, conf, method, votes = classify_defect_ensemble(f, image_key=image_key)
    regions, area_src = detect_areas(f, img_res, pred, img_filename, img_size=img_size, use_llm=use_llm_area)

    return {
        "image": img_res,
        "filename": img_filename,
        "true_label": true_label,
        "pred_label": pred,
        "confidence": conf,
        "cls_method": method,
        "votes": votes,
        "regions": regions,
        "area_method": area_src,
        "features": f
    }

print("Cell 7 OK — process_image() defined")


Cell 7 OK — process_image() defined


In [8]:

# ================================================================
# CELL 8 — VISUALIZATION + ANALYSIS HELPERS
# ================================================================
def serialize_prediction(p):
    q = {k:v for k,v in p.items() if k != "image"}
    return q

def visualize_batch(preds, model_name="LLM_Steel", batch_size=30, save=True, file_tag="detections"):
    total = len(preds)
    cols = 6
    for b in range(math.ceil(total/batch_size)):
        sl = preds[b*batch_size:(b+1)*batch_size]
        rows = math.ceil(len(sl)/cols)
        fig, axs = plt.subplots(rows, cols, figsize=(cols*3.5, rows*3.8))
        fig.patch.set_facecolor("#0d1117")
        flat = np.array(axs).reshape(-1) if hasattr(axs, "shape") else [axs]

        for i,pred in enumerate(sl):
            ax = flat[i]
            ax.imshow(pred["image"])
            ax.set_facecolor("#1c1c2e")
            for reg in pred.get("regions", []):
                x,y,bw,bh = reg["bbox"]
                lbl = reg.get("label", pred["pred_label"])
                color = DEFECT_COLORS.get(lbl, "#FF4444")
                ax.add_patch(Rectangle((x,y), bw, bh, linewidth=1.8, edgecolor=color, facecolor="none"))
                ax.text(x+1, max(y-2,0), f"{lbl[:4]} {pred['confidence']:.0%}",
                        fontsize=5, color="white", fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.1", facecolor=color, alpha=0.88))
            src_col = SOURCE_COLORS.get(pred.get("area_method",""), "#FFFFFF")
            true_txt = pred.get("true_label", "?")
            ok = "✓" if pred.get("true_label") == pred.get("pred_label") else "✗"
            ax.set_title(f"{ok} T:{true_txt}\nP:{pred['pred_label']} {pred['confidence']:.0%}\n{pred.get('area_method','?')}",
                         fontsize=6, color=src_col, fontweight="bold", pad=2)
            ax.axis("off")

        for j in range(len(sl), len(flat)):
            flat[j].set_visible(False)

        legend = [mpatches.Patch(color=c, label=n) for n,c in DEFECT_COLORS.items()]
        fig.legend(handles=legend, loc="lower center", ncol=6, fontsize=8,
                   facecolor="#161b22", labelcolor="white", bbox_to_anchor=(0.5,0))
        plt.tight_layout(rect=[0,0.05,1,1])

        if save:
            sp = os.path.join(OUTPUT_DIR, f"{model_name}_{file_tag}_batch_{b+1}.png")
            plt.savefig(sp, dpi=150, bbox_inches="tight", facecolor="#0d1117")
            print(f"Saved: {sp}")
        plt.show()

def create_area_analysis(predictions, model_name="LLM_Steel"):
    stats = {c: {"count":0, "conf":0, "regs":0, "area_pct":0} for c in CLASS_NAMES}
    for p in predictions:
        c = p["pred_label"]
        regs = p.get("regions", [])
        total_area = sum(r.get("area", r["bbox"][2]*r["bbox"][3]) for r in regs)
        stats[c]["count"] += 1
        stats[c]["conf"] += p.get("confidence",0)
        stats[c]["regs"] += len(regs)
        stats[c]["area_pct"] += total_area/(IMG_SIZE*IMG_SIZE)*100

    for c in CLASS_NAMES:
        n = max(stats[c]["count"], 1)
        stats[c]["conf"] /= n
        stats[c]["regs"] /= n
        stats[c]["area_pct"] /= n

    labels = CLASS_NAMES
    counts = [stats[c]["count"] for c in labels]
    confs = [stats[c]["conf"]*100 for c in labels]
    regs = [stats[c]["regs"] for c in labels]
    area = [stats[c]["area_pct"] for c in labels]

    fig, axes = plt.subplots(2,2, figsize=(16,10))
    fig.patch.set_facecolor("#0d1117")
    datasets = [(counts,"Predicted Count"), (confs,"Avg Confidence (%)"), (regs,"Avg Regions"), (area,"Avg Defect Area %")]
    for ax, (vals, title) in zip(axes.reshape(-1), datasets):
        ax.set_facecolor("#161b22")
        ax.bar(labels, vals, color=[DEFECT_COLORS[c] for c in labels], edgecolor="white", linewidth=0.5)
        ax.set_title(title, color="white", fontweight="bold")
        ax.tick_params(axis="x", rotation=35, colors="white")
        ax.tick_params(axis="y", colors="white")
        for sp in ax.spines.values(): sp.set_color("#30363d")
        ax.grid(axis="y", color="#21262d", linewidth=0.6)
    plt.tight_layout()
    sp = os.path.join(OUTPUT_DIR, f"{model_name}_area_analysis.png")
    plt.savefig(sp, dpi=150, bbox_inches="tight", facecolor="#0d1117")
    print(f"Saved: {sp}")
    plt.show()
    return stats

print("Cell 8 OK — visualization helpers defined")


Cell 8 OK — visualization helpers defined


In [ ]:

# ================================================================
# CELL 9 — RUN DATASET EVALUATION
# ================================================================
all_tasks = []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(IMAGES_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f"Missing dir: {cls_dir}")
        continue
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith((".jpg",".jpeg",".png",".bmp")):
            all_tasks.append({"cls":cls, "fname":fname, "path":os.path.join(cls_dir, fname)})

if LIMIT_IMAGES is not None:
    all_tasks = all_tasks[:LIMIT_IMAGES]

print(f"Total images found: {len(all_tasks)}")
print(f"Mode: {RUN_MODE} | Claude enabled: {claude_client is not None} | Groq enabled: {groq_client is not None}")

predictions = []
correct = 0
total = 0
class_acc = {c: {"correct":0, "total":0} for c in CLASS_NAMES}
cls_methods = Counter()
area_methods = Counter()
errors = []
run_start = time.time()

print("\n" + "="*74)
print("  RUNNING LLM-ONLY CLAUDE + GROQ EVALUATION")
print("="*74 + "\n")

for idx, task in enumerate(all_tasks, 1):
    try:
        result = process_image(task["path"], task["fname"], true_label=task["cls"],
                               use_llm_area=USE_LLM_AREA, img_size=IMG_SIZE)
        if "error" in result:
            errors.append({**task, "err": result["error"]})
            continue

        is_correct = result["pred_label"] == task["cls"]
        if is_correct:
            correct += 1
            class_acc[task["cls"]]["correct"] += 1

        total += 1
        class_acc[task["cls"]]["total"] += 1
        cls_methods[result["cls_method"]] += 1
        area_methods[result["area_method"]] += 1
        predictions.append(result)

    except Exception as ex:
        errors.append({**task, "err": str(ex)})
        continue

    if idx % PRINT_EVERY == 0 or idx == len(all_tasks):
        acc = correct/max(total,1)*100
        ela = time.time() - run_start
        eta = (ela/idx)*(len(all_tasks)-idx) if idx else 0
        icon = "🎯" if acc >= 95 else "📈"
        print(f"{icon} {idx:4d}/{len(all_tasks)} | Acc: {acc:6.2f}% | Correct: {correct}/{total} | "
              f"Elapsed: {ela/60:5.1f}m | ETA: {eta/60:5.1f}m")

    if idx % SAVE_EVERY == 0:
        with open(RESULTS_JSON, "w", encoding="utf-8") as f:
            json.dump([serialize_prediction(p) for p in predictions], f, indent=2)
        save_cache(LLM_CACHE)

final_acc = correct/max(total,1)*100
save_cache(LLM_CACHE)

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump([serialize_prediction(p) for p in predictions], f, indent=2)

with open(ERROR_JSON, "w", encoding="utf-8") as f:
    json.dump(errors, f, indent=2)

print("\n" + "="*74)
print(f"FINAL ACCURACY: {final_acc:.2f}%  ({correct}/{total})")
print("="*74)

print("\nPer-class accuracy:")
for c in CLASS_NAMES:
    ca = class_acc[c]["correct"]/max(class_acc[c]["total"],1)*100
    print(f"  {c:<18}: {ca:6.2f}%  ({class_acc[c]['correct']}/{class_acc[c]['total']})")

print("\nArea methods:", dict(area_methods))
print("Errors:", len(errors))
print("Saved results:", RESULTS_JSON)
print("Saved cache  :", CACHE_JSON)


Total images found: 1800
Mode: BALANCED | Claude enabled: True | Groq enabled: True

  RUNNING LLM-ONLY CLAUDE + GROQ EVALUATION

🎯   50/1800 | Acc: 100.00% | Correct: 50/50 | Elapsed:  39.5m | ETA: 1382.4m
🎯  100/1800 | Acc: 100.00% | Correct: 100/100 | Elapsed:  90.5m | ETA: 1538.7m
📈  150/1800 | Acc:  78.67% | Correct: 118/150 | Elapsed: 163.0m | ETA: 1793.4m


In [ ]:

# ================================================================
# CELL 10 — VISUALIZE DETECTIONS: 20 PER CLASS
# ================================================================
viz_sample = []
per_class_viz = {c:0 for c in CLASS_NAMES}
MAX_PER_CLASS = 20

for p in predictions:
    cls = p.get("true_label")
    if cls in per_class_viz and per_class_viz[cls] < MAX_PER_CLASS:
        viz_sample.append(p)
        per_class_viz[cls] += 1

print(f"Visualizing {len(viz_sample)} sample images...\n")
if viz_sample:
    visualize_batch(viz_sample, model_name=MODEL_NAME, batch_size=30, save=True, file_tag="sample_detections")
else:
    print("No predictions found. Run Cell 9 first.")


In [ ]:

# ================================================================
# CELL 11 — AREA-WISE ANALYSIS CHARTS
# ================================================================
if predictions:
    area_stats = create_area_analysis(predictions, model_name=MODEL_NAME)
    print("\n" + "─"*72)
    print(f'{"Class":<22} {"Count":>6} {"AvgConf":>9} {"AvgRegs":>9} {"AreaPct":>9}')
    print("─"*72)
    for cls in CLASS_NAMES:
        s = area_stats[cls]
        if s["count"] > 0:
            print(f'  {cls:<20} {s["count"]:>6} {s["conf"]:>8.1%} {s["regs"]:>9.2f} {s["area_pct"]:>8.1f}%')
else:
    print("Run Cell 9 first.")


In [ ]:

# ================================================================
# CELL 12 — ACCURACY SUMMARY + CONFUSION MATRIX
# ================================================================
if not predictions:
    print("Run Cell 9 first.")
else:
    BG = "#0d1117"; PAN = "#161b22"
    class_accs = [class_acc[c]["correct"]/max(class_acc[c]["total"],1)*100 for c in CLASS_NAMES]

    fig, axes = plt.subplots(1,2, figsize=(17,6))
    fig.patch.set_facecolor(BG)

    ax = axes[0]
    ax.set_facecolor(PAN)
    bars = ax.barh(CLASS_NAMES, class_accs, color=[DEFECT_COLORS[c] for c in CLASS_NAMES],
                   alpha=0.9, edgecolor="white", linewidth=0.5)
    ax.axvline(x=95, color="#00FF88", linewidth=2, linestyle="--", label="95% target")
    ax.axvline(x=final_acc, color="#FFA726", linewidth=2, linestyle="-.", label=f"Overall {final_acc:.1f}%")
    for bar, acc in zip(bars, class_accs):
        ax.text(min(acc+0.5,101), bar.get_y()+bar.get_height()/2,
                f"{acc:.1f}%", va="center", color="white", fontsize=9, fontweight="bold")
    ax.set_xlim(0,105)
    ax.set_xlabel("Accuracy (%)", color="white")
    ax.set_title("Per-Class Accuracy", color="white", fontweight="bold", fontsize=12)
    ax.tick_params(colors="white")
    for sp in ax.spines.values(): sp.set_color("#30363d")
    ax.legend(facecolor="#21262d", labelcolor="white", fontsize=9)
    ax.grid(axis="x", color="#21262d", linewidth=0.6)

    ax2 = axes[1]
    ax2.set_facecolor(PAN)
    provider_counts = Counter()
    for p in predictions:
        for v in p.get("votes", []):
            if v.get("provider") in ["claude","groq","rule_prior"]:
                provider_counts[v.get("provider")] += 1
    if provider_counts:
        labels = list(provider_counts.keys())
        values = [provider_counts[k] for k in labels]
        wedges, texts, autotexts = ax2.pie(values, labels=labels, autopct="%1.1f%%",
                                           textprops={"color":"white","fontsize":9}, startangle=140)
        for at in autotexts:
            at.set_color("#111")
    ax2.set_title("Vote Source Distribution", color="white", fontweight="bold", fontsize=12)

    plt.suptitle(f"{MODEL_NAME} — Overall: {final_acc:.2f}% ({correct}/{total}) | Target: 95%+",
                 fontsize=13, fontweight="bold", color="white")
    plt.tight_layout()
    sp = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_accuracy_summary.png")
    plt.savefig(sp, dpi=150, bbox_inches="tight", facecolor=BG)
    print(f"Saved: {sp}")
    plt.show()

    if SKLEARN_OK:
        y_true = [p["true_label"] for p in predictions]
        y_pred = [p["pred_label"] for p in predictions]
        cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)

        fig, ax = plt.subplots(figsize=(9,8))
        fig.patch.set_facecolor(BG)
        ax.set_facecolor(PAN)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title("Confusion Matrix", color="white", fontweight="bold")
        ax.tick_params(axis="x", rotation=35, colors="white")
        ax.tick_params(axis="y", colors="white")
        for text in ax.texts:
            text.set_fontweight("bold")
        sp = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_confusion_matrix.png")
        plt.savefig(sp, dpi=150, bbox_inches="tight", facecolor=BG)
        print(f"Saved: {sp}")
        plt.show()

        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, labels=CLASS_NAMES, digits=4))


In [ ]:

# ================================================================
# CELL 13 — MISCLASSIFIED SAMPLES ANALYSIS
# ================================================================
wrong = [p for p in predictions if p.get("true_label") != p.get("pred_label")]
print(f"Misclassified images: {len(wrong)}")

if wrong:
    rows = []
    for p in wrong[:30]:
        top_votes = Counter([v.get("label") for v in p.get("votes", []) if v.get("label")])
        rows.append({
            "filename": p["filename"],
            "true": p["true_label"],
            "pred": p["pred_label"],
            "confidence": round(p["confidence"],3),
            "top_votes": dict(top_votes.most_common(3)),
            "method": p["cls_method"]
        })
    wrong_df = pd.DataFrame(rows)
    display(wrong_df)

    visualize_batch(wrong[:min(30,len(wrong))], model_name=MODEL_NAME, batch_size=30,
                    save=True, file_tag="misclassified")
else:
    print("Excellent — no wrong predictions in current run.")


In [ ]:

# ================================================================
# CELL 14 — SINGLE IMAGE DEMO
# ================================================================
def demo_image(img_path, true_label=None):
    fname = os.path.basename(img_path)
    r = process_image(img_path, fname, true_label=true_label, use_llm_area=True, img_size=IMG_SIZE)
    if "error" in r:
        print(r["error"])
        return

    print(f"File: {fname}")
    print(f"True: {true_label}")
    print(f"Pred: {r['pred_label']} ({r['confidence']:.1%}) via {r['cls_method']}")
    print(f"Area: {r['area_method']} | Regions: {len(r['regions'])}")
    if true_label:
        print("Result:", "CORRECT" if r["pred_label"] == true_label else f"WRONG (true={true_label})")

    vote_rows = []
    for v in r.get("votes", []):
        vote_rows.append({
            "provider": v.get("provider"),
            "model": v.get("model"),
            "prompt": v.get("prompt","-"),
            "label": v.get("label"),
            "confidence": v.get("confidence"),
            "weight": v.get("weight"),
            "reason": v.get("reason","")[:90]
        })
    if vote_rows:
        display(pd.DataFrame(vote_rows))

    xml_reg = load_xml_annotations(fname, (IMG_SIZE, IMG_SIZE)) or []
    contour_reg = contour_regions(r["image"], r["pred_label"], IMG_SIZE)
    final_reg = r["regions"]

    sources = [
        ("Final Detection", final_reg, "#00FF88"),
        ("XML Ground Truth", xml_reg, "#4FC3F7"),
        ("OpenCV Contour", contour_reg, "#FFA726"),
    ]

    fig, axes = plt.subplots(1,3, figsize=(18,6))
    fig.patch.set_facecolor("#0d1117")
    for ax, (title, regs, tcol) in zip(axes, sources):
        ax.imshow(r["image"])
        ax.set_facecolor("#1c1c2e")
        for reg in regs:
            x,y,w,h = reg["bbox"]
            lbl = reg.get("label", r["pred_label"])
            color = DEFECT_COLORS.get(lbl, "#FF4444")
            ax.add_patch(Rectangle((x,y), w, h, linewidth=2.5, edgecolor=color, facecolor="none"))
            ax.text(x+2, y+14, lbl, fontsize=8, color="white", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85))
        ax.set_title(f"{title}\n{len(regs)} region(s)", color=tcol, fontweight="bold", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    sp = os.path.join(OUTPUT_DIR, f"demo_{os.path.splitext(fname)[0]}.png")
    plt.savefig(sp, dpi=150, bbox_inches="tight", facecolor="#0d1117")
    print(f"Saved: {sp}")
    plt.show()

# Example:
# demo_image(os.path.join(IMAGES_DIR, "scratches", "scratches_1.jpg"), true_label="scratches")
print("Cell 14 OK — demo_image() ready")
